# Week 2 · Class 2 — NumPy essentials & pandas fundamentals 🐼

**Plan for today (120 min)**

| Part | Topic | Time |
|---|---|---|
| 1 | NumPy essentials: arrays, shapes, operations, masks | 20 min |
| 2 | pandas fundamentals: Series, DataFrames, selecting, cleaning, grouping, combining | 90 min |
| 3 | Lambda functions & `.apply`: answering questions with custom rules | 10 min |

**Goal:** you won't learn *every* method today. You'll learn the **core vocabulary** of NumPy and pandas, so that you can
read the documentation, search for what you need, and solve Saturday's labs on your own.

After this class you should be able to:
- create NumPy arrays, check their `shape`, reshape/transpose them, compute statistics and label values with conditions
- create pandas `Series` and `DataFrame`s from lists and dictionaries, and read a CSV file
- select rows and columns (`[]`, `.loc`, `.iloc`, boolean filters) and compute statistics
- detect missing values, fix column types, set an index
- summarise data with `groupby`, and combine tables with `concat`, `merge` and `join`
- write `lambda` functions and use `.apply` to create new columns from your own rules

> 🧑‍💻 **Live-coding version** — we write the code together in class.

## 🧭 How to find what you don't know

Nobody remembers every method. Professionals use these tricks all the time:

- `np.zeros?` or `help(np.zeros)` → shows the documentation of a function inside the notebook
- Type `df.` and press **Tab** → lists every method available on your object
- Official docs: [NumPy reference](https://numpy.org/doc/stable/reference/index.html) · [pandas API reference](https://pandas.pydata.org/docs/reference/index.html)
- Searching "pandas how to ..." works very well — the answer is usually one method name away

---
# Part 1 — NumPy essentials (20 min) 🔢

**NumPy** is the library for working with numbers in Python. Its central object is the **array**: a grid of values that all have the **same type**.

Why not just Python lists?
- arrays are **much faster** for maths on many numbers
- you can do maths on the whole array at once (no loops needed)
- pandas is built on top of NumPy: every pandas column is a NumPy array underneath

In [1]:
# Import NumPy with its standard nickname "np"
import numpy as np

# Which version do we have?
print(np.__version__)

2.5.4


## 1.1 Creating arrays and inspecting them

The 4 attributes you'll check all the time:

| Attribute | Meaning |
|---|---|
| `.shape` | size along each dimension, e.g. `(2, 3)` = 2 rows, 3 columns |
| `.ndim` | number of dimensions (1 = vector, 2 = matrix, 3+ = tensor) |
| `.size` | total number of elements |
| `.dtype` | type of the elements (`int64`, `float64`, ...) |

In [13]:
# A 1D array (a vector) from a list
v = np.array([2,3,5])
print(v, v.shape, v.ndim, v.dtype)
# A 2D array (a matrix) from a list of lists: each inner list is a row
m = np.array([[1,2,3],
             [4,5,6]])
print(m)
print("shape", m.shape, "dimension:", m.ndim, "size:" , m.size)

[2 3 5] (3,) 1 int64
[[1 2 3]
 [4 5 6]]
shape (2, 3) dimension: 2 size: 6


### Ready-made arrays

Very often you don't type values by hand: you ask NumPy to build an array of a given **shape**.
The shape is always passed as a **tuple**, e.g. `(2, 3, 4)`.

In [15]:
# Arrays full of 0s or 1s
zeros = np.zeros((2,3))
print(zeros)
ones = np.ones((2,3,4))
print(ones)
# An "empty" array: memory is reserved but values are garbage until you fill them
# Regularly spaced values

[[0. 0. 0.]
 [0. 0. 0.]]
[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]


### Random arrays

Random data is great for practising and for simulations. The modern way is to create a **random generator** with a seed
(the seed makes the "random" numbers reproducible: same seed → same numbers).

In [21]:
# Create a random generator (seed = 42 makes results reproducible)
rng = np.random.default_rng(28)
# Uniform floats between 0 and 1, with a given shape
a = rng.random((2,3,5))
print(a)
# Random integers between 0 and 10 (10 excluded)
print(rng.integers(0,10, size=8))
# Normal distribution with mean 5 and standard deviation 2
print(rng.normal(5,2, size=5).round(3))
# You'll also see the older style in tutorials: it still works

[[[0.85183394 0.88489295 0.76570853 0.0321986  0.05019973]
  [0.32941503 0.87600378 0.95087951 0.53261165 0.7851304 ]
  [0.32933638 0.8180833  0.81674214 0.07272102 0.96257766]]

 [[0.34370794 0.17749607 0.51975146 0.62212851 0.0167896 ]
  [0.14867293 0.34406299 0.22183732 0.60308824 0.08558367]
  [0.73482412 0.47852248 0.2938055  0.96979051 0.98516564]]]
[2 9 7 9 8 0 9 3]
[5.437 5.987 5.354 5.304 5.716]


## 1.2 Indexing and slicing

Indexing starts at **0**. For a 2D array you write `array[row, column]`.
`:` means "everything", and `start:stop` takes a range (stop excluded).

In [35]:
# One element: row 1, column 3
b =np.arange(20).reshape(4,5)
print(b)
# A whole column
print(b[1,3])
# Rows 1 and 2, all columns
print(b[:,0])
print(b[1:3, :])
# Last two columns (negative indices count from the end)
print(b[:, -2])

[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
8
[ 0  5 10 15]
[[ 5  6  7  8  9]
 [10 11 12 13 14]]
[ 3  8 13 18]


In [36]:
list1 = [1,2, "Alex", "Margarida"]
list2 = [0, 3.14159, 98]
list1 + list2

[1, 2, 'Alex', 'Margarida', 0, 3.14159, 98]

## 1.3 Element-wise operations and broadcasting

Operations like `+`, `-`, `*`, `/` work **element by element**: no loop needed.

⚠️ To combine two arrays, their shapes must be **compatible**. Same shape always works.
A single number (a *scalar*) also works: NumPy "broadcasts" it to every element.

In [ ]:
x = np.array([1,2,3])
y = np.array([10,20,30])

print(x+y) # addition of vectors
print(x * y) # multiplication component wise of vectors
print(x@y) # dot product or the inner product

[11 22 33]
[10 40 90]
140


In [ ]:
# What happens with incompatible shapes?
# try/except lets us see the error without stopping the notebook

## 1.4 Reshape and transpose

- `.reshape(new_shape)` re-arranges the **same elements** into a new shape. The total size must stay the same. Use `-1` to let NumPy compute one dimension.
- `.T` swaps rows and columns of a matrix. For a 3D array it **reverses** all axes.
- `.transpose(axes)` lets you choose the **new order of the axes** yourself.

In [ ]:
# Transposing a 3D array: choose where each old axis goes
# We want shape (2, 3, 5): new axis order = old axes (1, 2, 0)

## 1.5 Statistics on arrays

`.sum()`, `.mean()`, `.min()`, `.max()`, `.std()` work on the whole array, or along one **axis**:
- `axis=0` → collapse the rows (one result per **column**)
- `axis=1` → collapse the columns (one result per **row**)

In [47]:
scores = np.array([[12,15,9], 
                   [18,14,16], 
                   [10,11,13]])
scores

print("overall scores mean", scores.mean().round(2))
print("mean per exam (axis=0)", scores.mean(axis=0))
print("mean per student (axis=1)", scores.mean(axis=1))
print("minimum score", scores.min())
print("maximum score", scores.max())

overall scores mean 13.11
mean per exam (axis=0) [13.33333333 13.33333333 12.66666667]
mean per student (axis=1) [12.         16.         11.33333333]
minimum score 9
maximum score 18


## 1.6 Masks and conditional labels

A comparison like `scores > 12` gives an array of `True`/`False` called a **mask**. You can use it to:
- **select** values: `scores[mask]`
- **count** them: `mask.sum()` (True counts as 1)
- **label** values: `np.where` (one condition) or `np.select` (several conditions)

These are the tools you need to label values in an array without writing loops.

In [ ]:
# np.where(condition, value_if_true, value_if_false)

In [ ]:
# np.select: list of conditions + list of matching values (first True wins)
# Same idea with numbers, and same shape as the input

### 🏋️ NumPy practice (3 min)

1. Create a random array `d` of shape `(2, 3, 4)` with values between 1 and 2 (hint: `rng.random` gives values in [0, 1)).
2. Store its min, max and mean in `d_min`, `d_max`, `d_mean`.
3. Create `f`, same shape as `d`, containing `0` where `d` equals its minimum, `100` where it equals its maximum, `25` below the mean and `75` above the mean.

In [ ]:
# your code here


---
# Part 2 — pandas fundamentals (90 min) 🐼

**pandas** is the library for **tables** (like Excel or SQL tables) in Python. Two objects to know:

| Object | What it is | Analogy |
|---|---|---|
| `Series` | one column of values with an **index** (labels) | one Excel column |
| `DataFrame` | several Series side by side, sharing the same index | a whole Excel sheet |

In [48]:
import pandas as pd
print(pd.__version__)

3.0.6


## 2.1 Series (8 min)

A Series is a 1D NumPy array **plus labels** (the index). By default the index is `0, 1, 2, ...`.

In [51]:
# Third value: position 2 (counting starts at 0)
temperatures = pd.Series([5.7, 75.2, 74.4, 84.0, 66.5])
print(temperatures)
print(temperatures.iloc[2])

0     5.7
1    75.2
2    74.4
3    84.0
4    66.5
dtype: float64
74.4


In [56]:
# A Series can have custom labels
prices = pd.Series([2.39, 3.45, 8.76], index =["chips", "drink", "burrito"])
print(prices)
# Statistics work directly
print(prices.loc["burrito"])
print(prices.mean(), prices.max(), prices.median())

chips      2.39
drink      3.45
burrito    8.76
dtype: float64
8.76
4.866666666666666 8.76 3.45


💡 **Rule of thumb:** `.loc` uses labels, `.iloc` uses integer positions. When in doubt, be explicit and use one of them.

## 2.2 Creating DataFrames (7 min)

The two most common ways to build a DataFrame by hand:
- from a **dictionary**: keys become column names, lists become columns
- from a **list of lists**: each inner list becomes a **row**

In [60]:
# From a dictionary of columns
people = pd.DataFrame({
    "name": ["Agnes", "Sidi", "Thibault", "Samia", "Henry", "George"],
    "age": [28, 37, 33, 28, 57, 29],
    "job": ["web analyst", "sales director", "web analyst", "sales director", "web analyst", "developer"]
})
people

,name,age,job
0,Agnes,28,web analyst
1,Sidi,37,sales director
2,Thibault,33,web analyst
3,Samia,28,sales director
4,Henry,57,web analyst
5,George,29,developer


In [ ]:
# From a list of lists: each sublist is a row
# Rename ALL columns at once by assigning a list
# Or rename only some columns with a dictionary {old: new}

In [68]:
# The key attributes of a DataFrame
print(people.shape)
print(people.columns)
print(people.index)
print(people.dtypes)
people.describe(include="all")

(6, 3)
Index(['name', 'age', 'job'], dtype='str')
RangeIndex(start=0, stop=6, step=1)
name      str
age     int64
job       str
dtype: object


,name,age,job
count,6,6.000000,6
unique,6,NaN,3
top,Agnes,NaN,web analyst
freq,1,NaN,3
mean,NaN,35.333333,NaN
std,NaN,11.183321,NaN
min,NaN,28.000000,NaN
25%,NaN,28.250000,NaN
50%,NaN,31.000000,NaN
75%,NaN,36.000000,NaN


## 2.3 Reading a file and first look (10 min)

We'll use a real dataset: orders from a restaurant chain (Chipotle). Each row is one item in an order.

`pd.read_csv` reads CSV files. This file is separated by **tabs**, so we pass `sep="\t"`.
It also works with a URL or a local path like `"data/chipotle.tsv"`.

In [73]:
#url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"
#orders = pd.read_csv(url, sep="\t")
orders = pd.read_csv("../data//chipotle.csv", index_col=0)
orders

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98
...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75


In [75]:
# First 5 rows (use .tail() for the last ones)
orders.head()
orders.tail()

,order_id,quantity,item_name,choice_description,item_price
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75
4621,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$8.75


In [78]:
# How big is it?
orders.shape
# Column names, types and number of non-missing values
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   order_id            4622 non-null   int64
 1   quantity            4622 non-null   int64
 2   item_name           4622 non-null   str  
 3   choice_description  3376 non-null   str  
 4   item_price          4622 non-null   str  
dtypes: int64(2), str(3)
memory usage: 180.7 KB


In [79]:
# Summary statistics of numeric columns
orders.describe()

,order_id,quantity
count,4622.000000,4622.000000
mean,927.254868,1.075725
std,528.890796,0.410186
min,1.000000,1.000000
25%,477.250000,1.000000
50%,926.000000,1.000000
75%,1393.000000,1.000000
max,1834.000000,15.000000


In [81]:
# For text columns: how many different values, and the most frequent ones
print(orders["item_name"].nunique())
orders["item_name"].value_counts()

50


item_name
Chicken Bowl                             726
Chicken Burrito                          553
Chips and Guacamole                      479
Steak Burrito                            368
Canned Soft Drink                        301
Chips                                    211
Steak Bowl                               211
Bottled Water                            162
Chicken Soft Tacos                       115
Chips and Fresh Tomato Salsa             110
Chicken Salad Bowl                       110
Canned Soda                              104
Side of Chips                            101
Veggie Burrito                            95
Barbacoa Burrito                          91
Veggie Bowl                               85
Carnitas Bowl                             68
Barbacoa Bowl                             66
Carnitas Burrito                          59
Steak Soft Tacos                          55
6 Pack Soft Drink                         54
Chips and Tomatillo Red Chili Salsa       48


## 2.4 Selecting data (15 min)

| You want | Syntax |
|---|---|
| one column (→ Series) | `df["col"]` |
| several columns (→ DataFrame) | `df[["col1", "col2"]]` ← note the **double brackets** |
| rows/columns by label | `df.loc[rows, cols]` |
| rows/columns by position | `df.iloc[rows, cols]` |
| rows matching a condition | `df[condition]` |

In [84]:
# One column → Series
people[["name", "age"]]

,name,age
0,Agnes,28
1,Sidi,37
2,Thibault,33
3,Samia,28
4,Henry,57
5,George,29


In [92]:
# Several columns → DataFrame (double brackets: a list inside [])
people

people.iloc[0:3, 0:2]

people.loc[1:3, ["age", "job"]]

,age,job
1,37,sales director
2,33,web analyst
3,28,sales director


In [ ]:
# .iloc: by position → first 3 rows, first 2 columns
# .loc: by label → rows with index 1 to 3 (INCLUDED with .loc!), columns age and job

### Boolean filtering

Exactly like NumPy masks: a condition gives a Series of `True`/`False`, and `df[condition]` keeps the `True` rows.

To combine conditions use `&` (and), `|` (or), `~` (not) — and **always put each condition in parentheses**.

In [94]:
# People older than 35
people[people["age"]>35]

,name,age,job
1,Sidi,37,sales director
4,Henry,57,web analyst


In [97]:
# Web analysts AND older than 28
mask = (people["job"] == "web analyst") & (people["age"]>28)
people[mask]

,name,age,job
2,Thibault,33,web analyst
4,Henry,57,web analyst


In [ ]:
# Several possible values: .isin()

In [ ]:
# Counting rows that match: len() of the filtered table, or .sum() of the mask
# A statistic on the filtered rows only

## 2.5 Cleaning basics (12 min)

Real data is messy. Before analysing, always check:
1. **missing values**
2. **column types** (numbers stored as text?)
3. **duplicates / identifiers**

In [99]:
# 1. Missing values per column
orders.isna().sum()

order_id                 0
quantity                 0
item_name                0
choice_description    1246
item_price               0
dtype: int64

In [100]:
# Two common options: drop the rows, or fill the gaps
orders["choice_description"] = orders["choice_description"].fillna("No choice")
orders.isna().sum()

order_id              0
quantity              0
item_name             0
choice_description    0
item_price            0
dtype: int64

In [101]:
# 2. item_price is text like "$2.39 " → we can't do maths with it
# Remove "$", remove spaces, convert to float
orders

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,No choice,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,No choice,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98
...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75


In [102]:
# 3. Setting a column as the index
# drop=False keeps the column in the table as well
orders["item_price"] = orders["item_price"].str.replace("$", "", regex=False).str.strip().astype(float)


In [104]:
orders["item_price"].describe()

count    4622.000000
mean        7.464336
std         4.245557
min         1.090000
25%         3.390000
50%         8.750000
75%         9.250000
max        44.250000
Name: item_price, dtype: float64

In [ ]:
# Is a column (or a combination of columns) a unique identifier?
# For several columns: look for duplicated combinations

## 2.6 New columns and statistics (6 min)

In [105]:
# Arithmetic between columns is element-wise (like NumPy)
# A custom rule on each value: .apply with a lambda function

orders

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,No choice,2.39
1,1,1,Izze,[Clementine],3.39
2,1,1,Nantucket Nectar,[Apple],3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,No choice,2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98
...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",8.75


In [106]:
# Column statistics: sum, mean, median, min, max
# Sorting
orders["unit_price"] = orders["item_price"]/ orders["quantity"]
orders

,order_id,quantity,item_name,choice_description,item_price,unit_price
0,1,1,Chips and Fresh Tomato Salsa,No choice,2.39,2.39
1,1,1,Izze,[Clementine],3.39,3.39
2,1,1,Nantucket Nectar,[Apple],3.39,3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,No choice,2.39,2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98,8.49
...,...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",11.75,11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",11.75,11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",11.25,11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",8.75,8.75


In [108]:
people["senior"] = people["age"].apply(lambda age : age >=40 )
people

,name,age,job,senior
0,Agnes,28,web analyst,False
1,Sidi,37,sales director,False
2,Thibault,33,web analyst,False
3,Samia,28,sales director,False
4,Henry,57,web analyst,True
5,George,29,developer,False


In [111]:
def senior(age):
    if age >=40:
        return True
    else:
        return False

people["senior_v1"] = people["age"].apply(senior)
people

,name,age,job,senior,senior_v1
0,Agnes,28,web analyst,False,False
1,Sidi,37,sales director,False,False
2,Thibault,33,web analyst,False,False
3,Samia,28,sales director,False,False
4,Henry,57,web analyst,True,True
5,George,29,developer,False,False


## 2.7 GroupBy: split → apply → combine (12 min)

`df.groupby("col")` splits the table into groups with the same value of `col`, applies a calculation to each group and combines the results.

In [113]:
# Average age per job
people.groupby("job")["age"].mean()

job
developer         29.000000
sales director    32.500000
web analyst       39.333333
Name: age, dtype: float64

In [ ]:
# Several statistics at once with .agg

In [ ]:
# Real data: top 5 items by revenue

In [115]:
# Revenue per order, then the average order value
orders.groupby("order_id")["item_price"].sum()

order_id
1       11.56
2       16.98
3       12.67
4       21.00
5       13.70
        ...  
1830    23.00
1831    12.90
1832    13.20
1833    23.50
1834    28.75
Name: item_price, Length: 1834, dtype: float64

## 2.8 Combining tables (12 min)

| Function | What it does | SQL analogy |
|---|---|---|
| `pd.concat([df1, df2])` | stacks tables **on top of each other** (or side by side with `axis=1`) | `UNION` |
| `pd.merge(left, right, on="key", how=...)` | matches rows using a **key column** | `JOIN` |
| `left.join(right)` | like merge, but matches on the **index** | `JOIN` on index |

In [116]:
sales_1 = pd.DataFrame({"sales_id": ["001", "002", "003", "004"],
                        "people_id": [1, 4, 2, 1],
                        "product_id": ["X789", "X999", "X789", "X990"]})
sales_2 = pd.DataFrame({"sales_id": ["005", "006", "007"],
                        "people_id": [0, 3, 2],
                        "product_id": ["X789", "X999", "X789"]})
products = pd.DataFrame({"product_id": ["X789", "X999", "X990"],
                         "product_desc": ["Apple", "Banana", "Orange"]})
customers = pd.DataFrame({"id": [0, 1, 2, 3, 5],
                          "name": ["Paul", "Perrine", "Moussa", "Michel", "Anne"],
                          "age": [None, 67, 24, 76, 47]})

In [117]:
sales_1

,sales_id,people_id,product_id
0,001,1,X789
1,002,4,X999
2,003,2,X789
3,004,1,X990


In [118]:
sales_2

,sales_id,people_id,product_id
0,005,0,X789
1,006,3,X999
2,007,2,X789


In [121]:
sales = pd.concat([sales_1, sales_2])
print(sales)

  sales_id  people_id product_id
0      001          1       X789
1      002          4       X999
2      003          2       X789
3      004          1       X990
0      005          0       X789
1      006          3       X999
2      007          2       X789


In [126]:
products

,product_id,product_desc
0,X789,Apple
1,X999,Banana
2,X990,Orange


In [127]:
sales_products = pd.merge(sales, products, on = "product_id")
sales_products

,sales_id,people_id,product_id,product_desc
0,001,1,X789,Apple
1,002,4,X999,Banana
2,003,2,X789,Apple
3,004,1,X990,Orange
4,005,0,X789,Apple
5,006,3,X999,Banana
6,007,2,X789,Apple


In [122]:
sales_1

,sales_id,people_id,product_id
0,001,1,X789
1,002,4,X999
2,003,2,X789
3,004,1,X990


In [119]:
products

,product_id,product_desc
0,X789,Apple
1,X999,Banana
2,X990,Orange


In [125]:
pd.concat([sales_1, products], axis=1)

,sales_id,people_id,product_id,product_id,product_desc
0,001,1,X789,X789,Apple
1,002,4,X999,X999,Banana
2,003,2,X789,X990,Orange
3,004,1,X990,NaN,NaN


In [120]:
customers

,id,name,age
0,0,Paul,NaN
1,1,Perrine,67.0
2,2,Moussa,24.0
3,3,Michel,76.0
4,5,Anne,47.0


In [128]:
sales_products

,sales_id,people_id,product_id,product_desc
0,001,1,X789,Apple
1,002,4,X999,Banana
2,003,2,X789,Apple
3,004,1,X990,Orange
4,005,0,X789,Apple
5,006,3,X999,Banana
6,007,2,X789,Apple


In [129]:
customers

,id,name,age
0,0,Paul,NaN
1,1,Perrine,67.0
2,2,Moussa,24.0
3,3,Michel,76.0
4,5,Anne,47.0


In [130]:
inner = pd.merge(sales_products, customers, left_on="people_id", right_on="id", how ="inner")
inner

,sales_id,people_id,product_id,product_desc,id,name,age
0,001,1,X789,Apple,1,Perrine,67.0
1,003,2,X789,Apple,2,Moussa,24.0
2,004,1,X990,Orange,1,Perrine,67.0
3,005,0,X789,Apple,0,Paul,NaN
4,006,3,X999,Banana,3,Michel,76.0
5,007,2,X789,Apple,2,Moussa,24.0


In [131]:
outer = pd.merge(sales_products, customers, left_on = "people_id", right_on= "id", how = "outer")
outer

,sales_id,people_id,product_id,product_desc,id,name,age
0,005,0.0,X789,Apple,0.0,Paul,NaN
1,001,1.0,X789,Apple,1.0,Perrine,67.0
2,004,1.0,X990,Orange,1.0,Perrine,67.0
3,003,2.0,X789,Apple,2.0,Moussa,24.0
4,007,2.0,X789,Apple,2.0,Moussa,24.0
5,006,3.0,X999,Banana,3.0,Michel,76.0
6,002,4.0,X999,Banana,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,5.0,Anne,47.0


In [132]:
left = pd.merge(sales_products, customers, left_on="people_id", right_on="id", how = "left")
left

,sales_id,people_id,product_id,product_desc,id,name,age
0,001,1,X789,Apple,1.0,Perrine,67.0
1,002,4,X999,Banana,NaN,NaN,NaN
2,003,2,X789,Apple,2.0,Moussa,24.0
3,004,1,X990,Orange,1.0,Perrine,67.0
4,005,0,X789,Apple,0.0,Paul,NaN
5,006,3,X999,Banana,3.0,Michel,76.0
6,007,2,X789,Apple,2.0,Moussa,24.0


In [ ]:
# Stack rows: same columns, one table under the other
# The index is repeated (0..3 then 0..2) → ignore_index=True renumbers it


In [ ]:
# axis=1 puts tables SIDE BY SIDE, aligning on the index

In [ ]:
# Merge on a common column name

In [134]:
orders 
## orders are cheap if the unit price < 5 , mid if btw 5 and 10, otherwise its a premium order

orders["price_band"] = orders["unit_price"].apply(lambda x: "cheap" if x < 5 else ("mid" if x < 10 else "premium" ))
orders

,order_id,quantity,item_name,choice_description,item_price,unit_price,price_band
0,1,1,Chips and Fresh Tomato Salsa,No choice,2.39,2.39,cheap
1,1,1,Izze,[Clementine],3.39,3.39,cheap
2,1,1,Nantucket Nectar,[Apple],3.39,3.39,cheap
3,1,1,Chips and Tomatillo-Green Chili Salsa,No choice,2.39,2.39,cheap
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98,8.49,mid
...,...,...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",11.75,11.75,premium
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",11.75,11.75,premium
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",11.25,11.25,premium
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",8.75,8.75,mid


### What if keys don't match?

`people_id` 4 has no customer, and customer 5 bought nothing. The `how` argument decides what to keep:

| `how=` | keeps |
|---|---|
| `"inner"` (default) | only keys present in **both** tables |
| `"left"` | all rows of the left table |
| `"right"` | all rows of the right table |
| `"outer"` | all rows of both, with `NaN` where data is missing |

When the key columns have **different names**, use `left_on=` and `right_on=`.

In [ ]:
# .join matches on the INDEX → set the key as index first

## 2.9 A taste of dates and text (5 min)

pandas has two "toolboxes" for special columns:
- `.dt` for dates and times (after converting with `pd.to_datetime`)
- `.str` for text (we already used `.str.replace` and `.str.strip`)

In [ ]:
# Text columns: .str methods apply to every value

In [ ]:
# Dates: text → datetime, then extract parts with .dt

---
# Part 3 — Lambda functions & `.apply` (10 min) λ

Sometimes pandas has no ready-made method for the rule you need. Then you write the rule as a **function** and ask pandas to
**apply** it to every value of a column.

### What is a lambda?

A `lambda` is a **tiny function without a name**, written in one line. These two are equivalent:

```python
def add_vat(price):
    return price * 1.19

add_vat = lambda price: price * 1.19
```

Read it as: *"take `price`, give back `price * 1.19`"*. Use `lambda` for short one-line rules, and `def` when the rule needs several lines.

In [ ]:
# A lambda is a normal function: you can store it and call it
# Lambdas can take several inputs

### `.apply` on a column

`df["col"].apply(function)` calls the function **once per value** and returns a new column with the results.

In [ ]:
# Question 1: how are orders split into price bands?

In [ ]:
# Question 2: which protein is the most popular in burritos, bowls and tacos?

### When the rule is longer: use `def`

`.apply` accepts **any** function, not only lambdas. If your rule needs several `if`s, a named function is easier to read.

In [ ]:
# Question 3: what share of the revenue comes from each category?

In [ ]:
# Question 4: do people pay more when they add guacamole?

### `.apply` with `groupby`: one rule per group

After a `groupby`, `.apply(lambda group: ...)` receives **all the values of one group** at once.

In [ ]:
# Question 5: what share of orders includes at least one drink?

💡 **Good habit:** `.apply` runs Python code row by row, so it is slower on big data. If a vectorised version exists
(arithmetic between columns, `.str.contains`, `np.where`...), prefer it. Use `.apply` when you need **custom logic**.

### 🏋️ Your turn (in class)

**A.** Create a column `n_toppings` with the number of choices in `choice_description` (hint: count the commas and add 1; `"No choice"` should give 0). What is the average `unit_price` for each number of toppings?

In [ ]:
# your code here


**B.** Using `order_totals` (revenue per order, from section 2.7), label each order `"small"` (< 15), `"medium"` (15–30) or `"large"` (> 30) with `.apply`. How many orders are in each group?

In [ ]:
# your code here


---
## 🏋️ Practice (at home or if time allows)

Use the `orders` table (already cleaned above).

**1.** How many rows have a `quantity` greater than 1 **and** an `item_price` above 10?

In [ ]:
# your code here


**2.** What is the average `item_price` of items whose name contains `"Bowl"`?

In [ ]:
# your code here


**3.** For each `item_name`, compute the total `quantity` sold. Which 3 items sold the most units?

In [ ]:
# your code here


**4.** Using `sales`, `products` and `customers`: build a table with one row per sale, the product description and the customer name. Keep all sales, even those without a known customer. How many sales have no customer name?

In [ ]:
# your code here


---
## 📋 Cheat sheet

**NumPy**
- create: `np.array`, `np.zeros`, `np.ones`, `np.empty`, `np.arange`, `np.linspace`, `rng = np.random.default_rng(seed)` → `rng.random`, `rng.integers`, `rng.normal`
- inspect: `.shape`, `.ndim`, `.size`, `.dtype`
- reshape: `.reshape`, `.T`, `.transpose(axes)`
- stats: `.sum()`, `.mean()`, `.min()`, `.max()`, `.std()` (with `axis=`)
- conditions: masks `a[a > x]`, `np.where`, `np.select`

**pandas**
- create/read: `pd.Series`, `pd.DataFrame`, `pd.read_csv`
- inspect: `.head()`, `.tail()`, `.shape`, `.info()`, `.describe()`, `.dtypes`, `.value_counts()`, `.nunique()`
- select: `df["col"]`, `df[["a", "b"]]`, `.loc`, `.iloc`, `df[cond]` with `&`, `|`, `~`, `.isin`
- clean: `.isna().sum()`, `.dropna()`, `.fillna()`, `.astype()`, `.rename()`, `.set_index()`, `.is_unique`, `.duplicated()`
- transform: column arithmetic, `.apply(lambda ...)` or `.apply(my_function)`, `groupby(...).apply(...)`, `.sort_values()`
- summarise: `.groupby().agg()`
- combine: `pd.concat`, `pd.merge(how=, on= / left_on= right_on=)`, `.join`
- special columns: `.str.*`, `pd.to_datetime`, `.dt.*`

**Next class:** we go deeper into exploratory data analysis and statistics (mean vs median, spread, outliers).